In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# PIPELINE FILE: p3_gold_pipeline
# PURPOSE      : DLT Gold layer — business-ready aggregates for dashboarding
#                from Silver Delta tables.
#
# TARGET:
#   catalog = restaurant_catalog
#   schema  = gold
#
# IMPORTANT:
#   This pipeline reads Silver tables from a DIFFERENT pipeline, so use
#   spark.read.table("catalog.schema.table") with fully qualified names.
#   These outputs are Gold materialized views for BI and interview demos.
# ═══════════════════════════════════════════════════════════════════════════════

import dlt
from pyspark.sql.functions import (
    col, sum as spark_sum, avg, count, round as spark_round,
    to_date, hour, explode, from_json, desc
)
from pyspark.sql.types import (
    ArrayType, StructType, StructField,
    StringType, IntegerType, DoubleType
)

CATALOG = "restaurant_catalog"
SILVER  = f"{CATALOG}.silver"

ITEMS_SCHEMA = ArrayType(StructType([
    StructField("item_id",    StringType(),  True),
    StructField("item_name",  StringType(),  True),
    StructField("category",   StringType(),  True),
    StructField("quantity",   IntegerType(), True),
    StructField("unit_price", DoubleType(),  True),
    StructField("subtotal",   DoubleType(),  True),
]))

In [0]:
# ─── GOLD 1: Daily revenue by restaurant ──────────────────────────────────────

@dlt.table(
    name="revenue_by_restaurant_daily",
    comment="Gold: daily revenue and order count by restaurant for BI dashboards.",
    table_properties={"quality": "gold"}
)
def revenue_by_restaurant_daily():
    orders = spark.read.table(f"{SILVER}.fact_orders")
    restaurants = spark.read.table(f"{SILVER}.dim_restaurants")

    return (
        orders.join(restaurants, "restaurant_id", "inner")
              .groupBy(
                  to_date(col("order_timestamp")).alias("order_date"),
                  col("restaurant_id"),
                  col("restaurant_name"),
                  col("city"),
                  col("cuisine"),
              )
              .agg(
                  count("order_id").alias("total_orders"),
                  spark_round(spark_sum("total_amount"), 2).alias("total_revenue_aed"),
                  spark_round(avg("total_amount"), 2).alias("avg_order_value_aed"),
              )
    )

In [0]:
# ─── GOLD 2: Orders by hour and restaurant ────────────────────────────────────

@dlt.table(
    name="orders_by_hour_restaurant",
    comment="Gold: hourly demand pattern by restaurant.",
    table_properties={"quality": "gold"}
)
def orders_by_hour_restaurant():
    orders = spark.read.table(f"{SILVER}.fact_orders")
    restaurants = spark.read.table(f"{SILVER}.dim_restaurants")

    return (
        orders.join(restaurants, "restaurant_id", "inner")
              .groupBy(
                  hour(col("order_timestamp")).alias("order_hour"),
                  col("restaurant_id"),
                  col("restaurant_name"),
                  col("city"),
              )
              .agg(
                  count("order_id").alias("total_orders"),
                  spark_round(spark_sum("total_amount"), 2).alias("total_revenue_aed"),
              )
    )

In [0]:
# ─── GOLD 3: Revenue by loyalty tier ──────────────────────────────────────────

@dlt.table(
    name="revenue_by_loyalty_tier",
    comment="Gold: revenue contribution by loyalty tier.",
    table_properties={"quality": "gold"}
)
def revenue_by_loyalty_tier():
    orders = spark.read.table(f"{SILVER}.fact_orders")
    customers = spark.read.table(f"{SILVER}.dim_customers")

    return (
        orders.join(customers, "customer_id", "left")
              .groupBy(col("loyalty_tier"))
              .agg(
                  count("order_id").alias("total_orders"),
                  spark_round(spark_sum("total_amount"), 2).alias("total_revenue_aed"),
                  spark_round(avg("total_amount"), 2).alias("avg_order_value_aed"),
              )
              .orderBy(desc("total_revenue_aed"))
    )

In [0]:
# ─── GOLD 4: Average rating by restaurant ─────────────────────────────────────

@dlt.table(
    name="avg_rating_by_restaurant",
    comment="Gold: average customer rating and review count by restaurant.",
    table_properties={"quality": "gold"}
)
def avg_rating_by_restaurant():
    reviews = spark.read.table(f"{SILVER}.fact_reviews")
    restaurants = spark.read.table(f"{SILVER}.dim_restaurants")

    return (
        reviews.join(restaurants, "restaurant_id", "inner")
               .groupBy(
                   col("restaurant_id"),
                   col("restaurant_name"),
                   col("city"),
               )
               .agg(
                   count("review_id").alias("total_reviews"),
                   spark_round(avg("rating"), 2).alias("avg_rating"),
               )
               .orderBy(desc("avg_rating"))
    )

In [0]:
# ─── GOLD 5: Top-selling categories ───────────────────────────────────────────
# items is a JSON array string in fact_orders, so parse and explode it.

@dlt.table(
    name="top_selling_categories",
    comment="Gold: category-level sales and quantity using exploded items JSON.",
    table_properties={"quality": "gold"}
)
def top_selling_categories():
    orders = spark.read.table(f"{SILVER}.fact_orders")
    restaurants = spark.read.table(f"{SILVER}.dim_restaurants")

    exploded_items = (
        orders.withColumn("item_array", from_json(col("items"), ITEMS_SCHEMA))
              .withColumn("item", explode(col("item_array")))
              .select(
                  col("restaurant_id"),
                  col("order_id"),
                  col("item.category").alias("category"),
                  col("item.quantity").alias("quantity"),
                  col("item.subtotal").alias("subtotal"),
              )
    )

    return (
        exploded_items.join(restaurants, "restaurant_id", "inner")
                      .groupBy(
                          col("restaurant_id"),
                          col("restaurant_name"),
                          col("city"),
                          col("category"),
                      )
                      .agg(
                          spark_sum("quantity").alias("total_quantity_sold"),
                          spark_round(spark_sum("subtotal"), 2).alias("total_category_revenue_aed"),
                      )
                      .orderBy(desc("total_category_revenue_aed"))
    )

In [0]:
# ─── GOLD 6: Payment method split ─────────────────────────────────────────────

@dlt.table(
    name="payment_method_split",
    comment="Gold: payment method distribution across all orders.",
    table_properties={"quality": "gold"}
)
def payment_method_split():
    orders = spark.read.table(f"{SILVER}.fact_orders")

    return (
        orders.groupBy("payment_method")
              .agg(
                  count("order_id").alias("total_orders"),
                  spark_round(spark_sum("total_amount"), 2).alias("total_revenue_aed"),
              )
              .orderBy(desc("total_orders"))
    )